# Pandas Hands-on Exercise - California Baby Names

### Dataset Introduction
This dataset (attached) contains information about **baby names** registered in California (USA) from 1910 to 2022.

#### Data Structure:
- **State**: State (CA = California)
- **Sex**: Gender (F = Female, M = Male)
- **Year**: Year of birth
- **Name**: Name
- **Count**: The number of babies with that name in that year

#### Example:
```
CA,F,2020,Olivia,2353  → In 2020, there were 2,353 baby girls named Olivia
CA,M,2020,Noah,2458    → In 2020, there were 2,458 baby boys named Noah
```

### Setup

In [25]:
import pandas as pd
import numpy as np

In [26]:
# Read the DataFrame from a CSV file, with the following columns
field_names = ['State', 'Sex', 'Year', 'Name', 'Count']
babynames_df = pd.read_csv('Data/STATE.CA.TXT',header=None,names=field_names)

In [27]:
# Show 10 rows randomly
babynames_df.head(10)

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
5,CA,F,1910,Ruth,128
6,CA,F,1910,Evelyn,126
7,CA,F,1910,Alice,118
8,CA,F,1910,Virginia,101
9,CA,F,1910,Elizabeth,93


**`Name popularity`**: Classification of Popularity Based on Count
- Count < 10: "Rare"
- 10 <= Count < 100: "Uncommon"
- 100 <= Count < 500: "Common"
- Count >= 500: "Very Popular"

### Task 1: Description basic information about this Dataset
**Instruction:**
- How many rows and how many columns does the DataFrame have?
- What is the data type for each column?
- How many unique names are in the dataset?
- From which year to which year does the dataset contain data?

In [28]:
print(f"Shape of DataFrame: {babynames_df.shape}")

print("\nData Types:")
print(babynames_df.dtypes)

print(f"\nUnique names: {babynames_df['Name'].nunique()}")

min_year = babynames_df['Year'].min()
max_year = babynames_df['Year'].max()
print(f"\nData range: {min_year} to {max_year}")

Shape of DataFrame: (407428, 5)

Data Types:
State    object
Sex      object
Year      int64
Name     object
Count     int64
dtype: object

Unique names: 20437

Data range: 1910 to 2022


### Task 2: Trend Analysis: analyzing how the Name "Emma" change over time
**Instruction**
- Get all records for the name "Emma"
- In which year did Emma first appear?
- In which year was Emma most popular
- How many baby boys were named Emma?
- Calculate the total number of babies named Emma from 1910 to 2022.

In [29]:
emma_df = babynames_df[babynames_df['Name'] == 'Emma']

emma_first_year = emma_df['Year'].min()
print(f"First appearance: {emma_first_year}")

# Total = Female + Male counts per year
emma_most_popular = emma_df.groupby('Year')['Count'].sum()
emma_most_popular_year = emma_most_popular.idxmax()
emma_most_popular_count = emma_most_popular.max()
print(f"Most popular year: {emma_most_popular_year} (Count: {emma_most_popular_count})")

emma_boys = emma_df[emma_df['Sex'] == 'M']['Count'].sum()
print(f"Baby boys named Emma: {emma_boys}")

total_emma = emma_df['Count'].sum()
print(f"Total babies named Emma: {total_emma}")


First appearance: 1910
Most popular year: 2018 (Count: 2751)
Baby boys named Emma: 5
Total babies named Emma: 57643


### Task 3: Analysis by Decade (1910s, 1920s, ..., 2020s)
**Instruction**
- Calculate the total number of babies born in each decade (create Decade column).
- Which decade had the highest number of babies?
- In the 1990s decade, how many unique names were there?
- "Very Popular" name mean Count >= 500. Compare the number of "Very Popular" names between the 1980s and 2010s decades.
- Calculate the average Count for each decade.

In [30]:
babynames_df['Decade'] = (babynames_df['Year'] // 10) * 10

decade_counts = babynames_df.groupby('Decade')['Count'].sum()
print("Total babies per decade:\n", decade_counts)
print(f"\nDecade with highest births: {decade_counts.idxmax()}s")

unique_names_1990s = babynames_df[babynames_df['Decade'] == 1990]['Name'].nunique()
print(f"\nUnique names in 1990s: {unique_names_1990s}")

def count_very_popular_name(decade):
    subset = babynames_df[babynames_df['Decade'] == decade]
    return subset[subset['Count'] >= 500]['Name'].nunique()

vp_1980s = count_very_popular_name(1980)
vp_2010s = count_very_popular_name(2010)
print(f"\nVery Popular names in 1980s: {vp_1980s}")
print(f"Very Popular names in 2010s: {vp_2010s}")

avg_count_decade = babynames_df.groupby('Decade')['Count'].mean()
print("\nAverage Count per decade:\n", avg_count_decade.round(3))


Total babies per decade:
 Decade
1910     289175
1920     683473
1930     768327
1940    1754798
1950    2994613
1960    3429746
1970    3075040
1980    4246867
1990    5048782
2000    4785634
2010    4246289
2020    1085487
Name: Count, dtype: int64

Decade with highest births: 1990s

Unique names in 1990s: 10453

Very Popular names in 1980s: 233
Very Popular names in 2010s: 252

Average Count per decade:
 Decade
1910     40.292
1920     55.662
1930     62.993
1940     98.662
1950    122.589
1960    111.882
1970     83.789
1980     86.210
1990     82.129
2000     70.623
2010     62.298
2020     55.535
Name: Count, dtype: float64


### Task 4: Analyze `Timeless` Names (Unchanging Over Time)
**Definition**: A name is called `timeless` if it:
- Appeared in **ALL** decades from the 1910s to the 2020s
- Always had a Count $\ge$ 100 in every decade

**Instruction**
- Find all "timeless" names.
- How many such names are there?
- Display the list of these names.
- Among these names, which one has the highest total Count?

In [31]:
all_decades = sorted(babynames_df['Decade'].unique())
num_decades = len(all_decades)

name_decade_counts = babynames_df.groupby(['Name', 'Decade'])['Count'].sum().reset_index()

# Group by Name to aggregate stats
timeless_stats = name_decade_counts.groupby('Name').agg(
    decades_present=('Decade', 'nunique'),
    min_count=('Count', 'min'),
    total_count=('Count', 'sum')
)

# Apply filters
timeless_names_df = timeless_stats[
    (timeless_stats['decades_present'] == num_decades) & 
    (timeless_stats['min_count'] >= 100)
]

print(f"Number of timeless names: {len(timeless_names_df)}")
print("\nList of timeless names:", list(timeless_names_df.index))

most_popular_timeless = timeless_names_df['total_count'].idxmax()
print(f"\nTimeless name with highest total count: {most_popular_timeless}")


Number of timeless names: 206

List of timeless names: ['Ada', 'Aileen', 'Alan', 'Albert', 'Alex', 'Alexander', 'Alfonso', 'Alice', 'Allan', 'Allen', 'Alma', 'Alvin', 'Amelia', 'Amy', 'Andrew', 'Angela', 'Angelina', 'Angelo', 'Angie', 'Anna', 'Annie', 'Anthony', 'Antonio', 'Arthur', 'Audrey', 'Aurora', 'Barbara', 'Beatrice', 'Ben', 'Benjamin', 'Billie', 'Bonnie', 'Bruce', 'Carlos', 'Carmen', 'Caroline', 'Catherine', 'Cecilia', 'Celia', 'Charles', 'Charlie', 'Charlotte', 'Christine', 'Claire', 'Clara', 'Cora', 'Daniel', 'David', 'Donald', 'Dorothy', 'Eddie', 'Edgar', 'Edith', 'Edward', 'Edwin', 'Eileen', 'Elaine', 'Eleanor', 'Elizabeth', 'Ella', 'Elsie', 'Emily', 'Emma', 'Esther', 'Eugene', 'Eva', 'Evelyn', 'Everett', 'Felix', 'Flora', 'Forrest', 'Frances', 'Francis', 'Francisco', 'Frank', 'Franklin', 'Frederick', 'Genevieve', 'George', 'Georgia', 'Gloria', 'Grace', 'Guadalupe', 'Gwendolyn', 'Harry', 'Harvey', 'Hazel', 'Helen', 'Henry', 'Irene', 'Isabel', 'Jack', 'Jacqueline', 'James', 

### Task 5: `Your Name` Analysis
**Perform an in-depth analysis of any given name**\
Create a function `analyze_name(name)` that performs the following:

1. Check if the name exists in the dataset.
2. If it exists, display:
   - The first year it appeared.
   - The most popular year (and its Count).
   - The total number of people with this name.
   - Which gender uses this name more frequently.
   - Which category this name belongs to (Rare/Uncommon/Common/Very Popular) in the last 5 years.
     
***Test with these name:***
```
['Emma', 'Noah', 'Michael', 'Sophia', 'James']
```

In [32]:
def analyze_name(name):
    name_df = babynames_df[babynames_df['Name'] == name]
    print(f"Analysis for: {name}")

    if name_df.empty:
        print(f"The name '{name}' does not exist in the dataset")
        return

    name_first_year = name_df['Year'].min()
    total_people = name_df['Count'].sum()

    yearly_sum = name_df.groupby('Year')['Count'].sum()
    popular_year = yearly_sum.idxmax()
    popular_count = yearly_sum.max()
    
    print(f"First appeared: {name_first_year}")
    print(f"Most popular year: {popular_year} (Count: {popular_count})")
    print(f"Total Count: {total_people}")
    
    sex_counts = name_df.groupby('Sex')['Count'].sum()
    if len(sex_counts) > 1:
        if sex_counts.get('F', 0) > sex_counts.get('M', 0):
            dominant_sex = "Female"
        else:
            dominant_sex = "Male"
    else:
        dominant_sex = "Female" if 'F' in sex_counts else "Male"
        
    print(f"Most frequent gender: {dominant_sex}")
    
    lastest_year = babynames_df['Year'].max()
    last_5_years_df = name_df[name_df['Year'] >= (lastest_year - 4)]
    
    if last_5_years_df.empty:
        print("Category (Last 5 years): Not appeared")
    else:
        avg_last_5_years_count = last_5_years_df.groupby('Year')['Count'].sum().mean()
        
        category = "Rare"
        if avg_last_5_years_count >= 500:
            category = "Very Popular"
        elif avg_last_5_years_count >= 100:
            category = "Common"
        elif avg_last_5_years_count >= 10:
            category = "Uncommon"
            
        print(f"Category (Last 5 years avg): {category} (Avg Count: {avg_last_5_years_count:.3f})")
    print("\n")


test_names = ['Emma', 'Noah', 'Michael', 'Sophia', 'James']
for n in test_names:
    analyze_name(n)


Analysis for: Emma
First appeared: 1910
Most popular year: 2018 (Count: 2751)
Total Count: 57643
Most frequent gender: Female
Category (Last 5 years avg): Very Popular (Avg Count: 2304.600)


Analysis for: Noah
First appeared: 1942
Most popular year: 2014 (Count: 2796)
Total Count: 58126
Most frequent gender: Male
Category (Last 5 years avg): Very Popular (Avg Count: 2627.000)


Analysis for: Michael
First appeared: 1910
Most popular year: 1990 (Count: 8314)
Total Count: 435716
Most frequent gender: Male
Category (Last 5 years avg): Very Popular (Avg Count: 1158.600)


Analysis for: Sophia
First appeared: 1910
Most popular year: 2012 (Count: 3644)
Total Count: 63319
Most frequent gender: Female
Category (Last 5 years avg): Very Popular (Avg Count: 1967.400)


Analysis for: James
First appeared: 1910
Most popular year: 1957 (Count: 5500)
Total Count: 286322
Most frequent gender: Male
Category (Last 5 years avg): Very Popular (Avg Count: 1254.200)


